In [5]:
# загружаем стиль для оформления файла
from IPython.display import HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

<h1><u>План урока</u></h1>

<p><font size="3" face="Arial">
<ul type="square">
    <a href="#1"><li>Предобработка данных (Data Preprocessing / Preparation)</li></a>
    <a href="#2"><li>Разделы предобработки данных</li></a>
    <a href="#3"><li>Переименования</li></a>
    <a href="#4"><li>Пропуски</li></a>
    <a href="#5"><li>Нормировка</li></a>
    <a href="#6"><li>Обработка категориальных значений</li></a>
    <a href="#7"><li>Target Encoding</li></a>
    <a href="#8"><li>Итог + дополнительный материалы</li></a>
</ul></font></p>

<h2><b>Предобработка данных (Data Preprocessing / Preparation)</b></h2>

Четкого и устоявшего определения не существует, поэтому приведем пример нескольких наиболее ёмких.

**Предобработка данных**:    
  * замена, модификация или удаление частей набора данных с целью повышения непротиворечивости, полноты, корректности и ясности набора данных, а также уменьшения избыточности
  * процесс преобразования данных в форму, удобную для анализа

Выполняется на полном наборе данных (и на контрольных объектах тоже).

<h2><b>Разделы предобработки данных</b></h2>

1. Трансформация данных (Data Transformation)
    - Переименование признаков, объектов, значений признаков, преобразование типов
    - Кодирование значений категориальных переменных
    - Дискретизация (Discretization / Binning)
    - Нормализация (Normalization)
    - Сглаживание (Smoothing)
    - Создание признаков (Feature creation)
    - Агрегирование (Aggregation)
    - Обобщение (Generalization)
    - Деформация значений


2. Интеграция данных (Data Integration)
    - Объединение данных из разных источников

<h2><b>Переименования</b></h2>

Названия переменных должны быть интуитивны (они используются в том числе при передачи данных коллегам, презентации результатов и т.п.).

<h2><b>Пропуски — как выглядят в данных</b></h2>

1. Пустые значения
2. Специальные значения (NA, NaN, null, ...)
3. Специальный код (–999, mean, число за пределами значения признака)

In [6]:
import seaborn as sns
df = sns.load_dataset("titanic")
df.head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


In [7]:
print(df['deck'].isnull().sum()) # число "нанов"
print(df['deck'].count()) # число не "нанов"

688
203


**Пропуски** — что делать:

- оставляем (но не все модели могут работать с пропусками)
- удаляем описания объектов с пропусками / признаки (радикальная мера, которая редко используется). Важно не удалить большой процент данных

``` python
df.dropna(how='any', axis=1)
```

- заменяем на фиксированное значение (например, если признак бинарный, то на 0.5). Значение –999, как правило, плохое – является выбросом

``` python
df.fillna(-1)
```

- заменяем на легковычислимое значение (среднее, медиана, мода)

``` python
df.fillna(df.mean()) # , inplace=True
```

- восстановление значения (построение специальной модели для восстановления)

``` python
from sklearn.preprocessing import Imputer

imputer = Imputer(missing_values='NaN', strategy='mean', axis=0)
vals = imputer.fit_transform(df[name])
```

- экспертная замена (нужно четко понимать природу данных, иметь экспертизу в области)

- во временных рядах пропуски можно заполнить интерполяцией

``` python
ts = pd.Series(vals, index=index)
ts2 = ts.interpolate()
ts3 = ts.interpolate(method='time')
ts4 = ts.interpolate(method='polynomial', order=2)
```

- добавлять характеристический признак пропусков «is_nan». Тогда модель сама определит оптимальное значение для заполнения

Заполнять пропуски лучше после генерации признаков, иначе возникают дополнительные неопределённости. Можно посмотреть, зависит ли факт пропуска от других данных.

<h2><b>Нормировка Data Normalization</b></h2>

Желательно, чтобы все признаки датасета были вещественными и в одной шкале.

1. Стандартизация (Z-score Normalization/Variance Scaling)

$$ \frac{u_i-mean(u_t)}{std(u_t)}$$

2. Нормировка на отрезок (Min-Max Normalization)

$$ \frac{u_i-min(u_t)}{max(u_t)-min(u_t)}$$

3. Нормировка по максимуму

$$ \frac{u_i}{max(u_t)}$$

4. Decimal Scaling Normalization
5. Ранговая нормировка (tiedrank, rankdata)

In [8]:
import numpy as np
# Нормировка производится по колонкам, например:

X = df[['age']]

X = X / np.max(X)
X = X - np.min(X)
X = X / np.max(X)
X = X - np.mean(X)
X = X / np.std(X)

from scipy.stats import rankdata
X = rankdata(X, method='average')

X = X - np.min(X)
X = X / np.max(X)

import sklearn.preprocessing as prp
X['minmax'] = prp.minmax_scale(X['name'])
X['standart'] = prp.StandardScaler().fit_transform(X[['name']]) # требует df

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

Реализуем на нашем датафрейме.

In [9]:
import numpy as np

# 1. Стандартизация (Z-score Normalization)
df['age_zscore'] = (df['age'] - df['age'].mean()) / df['age'].std()

# Стандартизация при помощи библиотеки sklearn
import sklearn.preprocessing as prp
from sklearn.preprocessing import StandardScaler
df['age_zscore_sklearn'] = StandardScaler().fit_transform(df[['age']]) # требует df

# 2. Нормировка на отрезок (Min-Max)
df['age_minmax'] = (df['age'] - df['age'].min()) / (df['age'].max() - df['age'].min())

# 3. Нормировка по максимуму
df['age_maxnorm'] = df['age'] / df['age'].max()

# Decimal Scaling Normalization
def decimal_scaling(series):
    max_abs = np.abs(series).max()
    j = int(np.ceil(np.log10(max_abs + 1)))
    return series / (10 ** j)

df['age_decimal'] = decimal_scaling(df['age'])

# Ранговая нормировка
# rankdata() не умеет корректно обрабатывать NaN, поэтому сперва заполняем пропуски (например, медианой)
from scipy.stats import rankdata

df['age'] = df['age'].fillna(df['age'].median())

df['age_rank'] = rankdata(df['age'], method='average')
df['age_rank_norm'] = df['age_rank'] / df['age_rank'].max()
df['age_tiedrank'] = rankdata(df['age'], method='average')

df.head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,alive,alone,age_zscore,age_zscore_sklearn,age_minmax,age_maxnorm,age_decimal,age_rank,age_rank_norm,age_tiedrank
0,0,3,male,22.0,1,0,7.2500,S,Third,man,...,no,False,-0.530005,-0.530377,0.271174,0.2750,0.22,218.0,0.244669,218.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,...,yes,False,0.571430,0.571831,0.472229,0.4750,0.38,709.0,0.795735,709.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,...,yes,True,-0.254646,-0.254825,0.321438,0.3250,0.26,310.5,0.348485,310.5
3,1,1,female,35.0,1,0,53.1000,S,First,woman,...,yes,False,0.364911,0.365167,0.434531,0.4375,0.35,665.5,0.746914,665.5
4,0,3,male,35.0,0,0,8.0500,S,Third,man,...,no,True,0.364911,0.365167,0.434531,0.4375,0.35,665.5,0.746914,665.5
5,0,3,male,28.0,0,0,8.4583,Q,Third,man,...,no,True,NaN,NaN,NaN,NaN,NaN,438.5,0.492144,438.5
6,0,1,male,54.0,0,0,51.8625,S,First,man,...,no,True,1.672866,1.674039,0.673285,0.6750,0.54,845.5,0.948934,845.5
7,0,3,male,2.0,3,1,21.0750,S,Third,child,...,no,False,-1.906799,-1.908136,0.019854,0.0250,0.02,19.5,0.021886,19.5
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,...,yes,False,-0.185807,-0.185937,0.334004,0.3375,0.27,328.5,0.368687,328.5
9,1,2,female,14.0,1,0,30.0708,C,Second,child,...,yes,False,-1.080723,-1.081480,0.170646,0.1750,0.14,74.5,0.083614,74.5


Нормировки в пределах группы.

In [10]:
z_score = lambda x: (x - x.mean()) / x.std()
df['var'] = df.groupby('group').transform(z_score)

KeyError: 'group'

Применим к нашим данным.

In [12]:
# Нормировка возраста по группам выживания
z_score = lambda x: (x - x.mean()) / x.std()
df['age_survived'] = df.groupby('survived')['age'].transform(z_score)
df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,alone,age_zscore,age_zscore_sklearn,age_minmax,age_maxnorm,age_decimal,age_rank,age_rank_norm,age_tiedrank,age_survived
0,0,3,male,22.0,1,0,7.2500,S,Third,man,...,False,-0.530005,-0.530377,0.271174,0.2750,0.22,218.0,0.244669,218.0,-0.642259
1,1,1,female,38.0,1,0,71.2833,C,First,woman,...,False,0.571430,0.571831,0.472229,0.4750,0.38,709.0,0.795735,709.0,0.705338
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,...,True,-0.254646,-0.254825,0.321438,0.3250,0.26,310.5,0.348485,310.5,-0.166475
3,1,1,female,35.0,1,0,53.1000,S,First,woman,...,False,0.364911,0.365167,0.434531,0.4375,0.35,665.5,0.746914,665.5,0.487384
4,0,3,male,35.0,0,0,8.0500,S,Third,man,...,True,0.364911,0.365167,0.434531,0.4375,0.35,665.5,0.746914,665.5,0.397742


Ранговая нормировка.

In [13]:
import scipy.stats as ss

for method in ['average', 'min', 'max', 'dense', 'ordinal']:
    data[method] = ss.rankdata(data.feat, method=method)

NameError: name 'data' is not defined

Применим к нашим данным.

In [14]:
import scipy.stats as ss

for method in ['average', 'min', 'max', 'dense', 'ordinal']:
    df[method] = ss.rankdata(df['age'], method=method)
df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,age_decimal,age_rank,age_rank_norm,age_tiedrank,age_survived,average,min,max,dense,ordinal
0,0,3,male,22.0,1,0,7.2500,S,Third,man,...,0.22,218.0,0.244669,218.0,-0.642259,218.0,205.0,231.0,29.0,205.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,...,0.38,709.0,0.795735,709.0,0.705338,709.0,704.0,714.0,52.0,704.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,...,0.26,310.5,0.348485,310.5,-0.166475,310.5,302.0,319.0,35.0,302.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,...,0.35,665.5,0.746914,665.5,0.487384,665.5,657.0,674.0,48.0,657.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,...,0.35,665.5,0.746914,665.5,0.397742,665.5,657.0,674.0,48.0,658.0


<h2><b>Обработка категориальных значений</b></h2>

Простейшее кодирование – по номеру категории **Label Encoding**

- Лексикографический порядок (sklearn)

``` python
sklearn.preprocessing.LabelEncoder
```

- В порядке появления (pandas)

``` python
pandas.factorize
```

\- не подходит для линейных алгоритмов

\- проблема новых категорий

In [15]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

for name in cols:
    # лексикографический порядок
    data[name + '_le'] = le.fit_transform(data[name])

    # в порядке упоминания
    data[name + '_fz'] = pd.factorize(data[name])[0]

    # случайно
    dct = dict(zip(data[name].unique(), np.random.rand(data[name].nunique()).round(2)))
    data[name + '_rnd'] = data[name].map(dct)

NameError: name 'cols' is not defined

Применим к нашим данным.

In [ ]:
import pandas as pd
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

for name in df.columns[1:15]: # применим к изначальным переменным
    # лексикографический порядок
    df[name + '_le'] = le.fit_transform(df[name])

    # в порядке упоминания
    df[name + '_fz'] = pd.factorize(df[name])[0]

    # случайно
    dct = dict(zip(df[name].unique(), np.random.rand(df[name].nunique()).round(2)))
    df[name + '_rnd'] = df[name].map(dct)

df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,deck_rnd,embark_town_le,embark_town_fz,embark_town_rnd,alive_le,alive_fz,alive_rnd,alone_le,alone_fz,alone_rnd
0,0,3,male,22.0,1,0,7.2500,S,Third,man,...,0.69,2,0,0.53,0,0,0.65,0,0,0.69
1,1,1,female,38.0,1,0,71.2833,C,First,woman,...,0.12,0,1,0.28,1,1,0.92,0,0,0.69
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,...,0.69,2,0,0.53,1,1,0.92,1,1,0.87
3,1,1,female,35.0,1,0,53.1000,S,First,woman,...,0.12,2,0,0.53,1,1,0.92,0,0,0.69
4,0,3,male,35.0,0,0,8.0500,S,Third,man,...,0.69,2,0,0.53,0,0,0.65,1,1,0.87


**Dummy-кодирование / One-hot-encoding**

\+ подходит для линейных алгоритмов

\+ можно кодировать N-1 категорию

\- большое число категорий

\- сильно разреженные матрицы

Плохо, что после OHE значительная часть признаков – бинарные. Некоторые алгоритмы (RF) могут терять качество.

Плохо, что значительная часть – признаки с большим числом категорий.

In [ ]:
from sklearn import preprocessing
ohe = preprocessing.OneHotEncoder(sparse=False)
tmp = ohe.fit_transform(data[cols]).astype(int)
tmp = pd.DataFrame(tmp, columns=['OHE_' + str(i) for i in range(tmp.shape[1])])
data = pd.concat([data, tmp], axis=1)

# Ручное решение
def code_myohe(data, feature):
    """
    ручной способ OHE
    """
    for i in data[feature].unique():
        data[feature + '=' + i] = (data[feature] == i).astype(int)

for name in cols:
    code_myohe(data, name)

#Самый простой способ
pd.get_dummies(data)

Применим к нашим данным.

Загрузим датасет заново для простоты.

In [ ]:
df = sns.load_dataset("titanic")
df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Для обработки возьмем категориальную переменную class.

In [ ]:
from sklearn import preprocessing
import pandas as pd

ohe = preprocessing.OneHotEncoder(sparse_output=False)  # берет на вход двумерный массив
tmp = ohe.fit_transform(df[['class']])

tmp = pd.DataFrame(
    tmp,
    columns=['OHE_' + str(i) for i in range(tmp.shape[1])],
    index=df.index
)

data = pd.concat([df, tmp], axis=1)
data.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,OHE_0,OHE_1,OHE_2
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,0.0,0.0,1.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,1.0,0.0,0.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,0.0,0.0,1.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,1.0,0.0,0.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,0.0,0.0,1.0


**Дискретизация (биннинг / Binning, квантование / Quantization)** – переход от вещественного признака к порядковому за счёт кодирования интервалов одним значением.

Например: доход от 0 до 10000, от 10000 до 25000, от 25000 до 50000 и т.д.

\+ улучшает интерпретацию

\+ позволяет решать задачу простыми алгоритмами

\- качество ухудшается

In [ ]:
bins = pd.cut(df[name], 5)

points = [0, 12, 18, 25, 50, 100]
labels = ['ребёнок', 'юноша', 'молодой человек', 'мужчина', 'пожилой']
factors = pd.cut(ages, points, labels=labels)
factors.describe()

Применим к нашим данным.

In [ ]:
points = [0, 12, 18, 25, 50, 100]
labels = ['ребёнок', 'подросток', 'молодой', 'взрослый', 'пожилой']

df['age_group'] = pd.cut(
    df['age'],
    bins=points,
    labels=labels,
    include_lowest=True
)
df.head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,age_group
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,молодой
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,взрослый
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,взрослый
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,взрослый
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,взрослый
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True,NaN
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True,пожилой
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False,ребёнок
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False,взрослый
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False,подросток


Зададим одномерный массив.

In [ ]:
ages = df['age']

**Способы дискретизации**

**Equal-width (distance) partitioning**

Делим область значения признаков на области — интервалы равной длины

0 – 9, 10 – 99, 100 – 999

In [ ]:
factors = pd.cut(ages, 4)
factors

0      (20.315, 40.21]
1      (20.315, 40.21]
2      (20.315, 40.21]
3      (20.315, 40.21]
4      (20.315, 40.21]
            ...       
886    (20.315, 40.21]
887     (0.34, 20.315]
888                NaN
889    (20.315, 40.21]
890    (20.315, 40.21]
Name: age, Length: 891, dtype: category
Categories (4, interval[float64, right]): [(0.34, 20.315] < (20.315, 40.21] < (40.21, 60.105] < (60.105, 80.0]]

**Equal-depth (frequency, Quantile-based) partitioning**

Делим область значения признаков на области – интервалы: в каждую попало одинаковое число точек.

In [ ]:
factors = pd.qcut(ages, 4)
factors

0       (20.125, 28.0]
1         (28.0, 38.0]
2       (20.125, 28.0]
3         (28.0, 38.0]
4         (28.0, 38.0]
            ...       
886     (20.125, 28.0]
887    (0.419, 20.125]
888                NaN
889     (20.125, 28.0]
890       (28.0, 38.0]
Name: age, Length: 891, dtype: category
Categories (4, interval[float64, right]): [(0.419, 20.125] < (20.125, 28.0] < (28.0, 38.0] < (38.0, 80.0]]

Также можно использовать модели кластеризации и экспертный метод.

**Хэш-кодирование**

Cредство против сильно разреженных данных.

Могут быть коллизии (можно выполнять разные хэш-кодирования).

In [ ]:
from sklearn.feature_extraction import FeatureHasher
fh = FeatureHasher(n_features=2, input_type='string')

for name in cols:
    tmp = fh.fit_transform(data[name]).toarray()
    tmp = pd.DataFrame(tmp, columns=[name + '_' + str(i) for i in range(tmp.shape[1])])
    data = pd.concat([data, tmp], axis=1)

Применим к нашим данным.

In [ ]:
from sklearn.feature_extraction import FeatureHasher
fh = FeatureHasher(n_features=2, input_type='string')

hashed_features = []

for name in df.select_dtypes(include='object').columns:

    values = df[name].astype(str).apply(lambda x: [x])
    tmp = fh.transform(values).toarray()

    tmp = pd.DataFrame(
        tmp,
        columns=[name + '_' + str(i) for i in range(tmp.shape[1])],
        index=df.index
    )

    hashed_features.append(tmp)

df = pd.concat([df] + hashed_features, axis=1)
df.head(5)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,sex_0,sex_1,embarked_0,embarked_1,who_0,who_1,embark_town_0,embark_town_1,alive_0,alive_1
0,0,3,male,22.0,1,0,7.2500,S,Third,man,...,-1.0,0.0,-1.0,0.0,1.0,0.0,0.0,-1.0,1.0,0.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,...,0.0,-1.0,0.0,-1.0,1.0,0.0,1.0,0.0,-1.0,0.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,...,0.0,-1.0,-1.0,0.0,1.0,0.0,0.0,-1.0,-1.0,0.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,...,0.0,-1.0,-1.0,0.0,1.0,0.0,0.0,-1.0,-1.0,0.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,...,-1.0,0.0,-1.0,0.0,1.0,0.0,0.0,-1.0,1.0,0.0


Проблема мелких и новых категорий возникает почти для всех способов кодирования.

Часто мелкие категории объединить в одну.

<h2><b>Target Encoding</b></h2>

Упорядочивание категорий исходя из смысла задачи.

**Проблемы**:
- кодировка мелких категорий
- слияние мелких категорий
- нельзя допустить утечки значений целевого признака

**Сглаживание**

Добавляется среднее значение целевого признака с весом. Борьба с редкими категориями, на них оценка ненадёжна.

$mean=\frac{k_1+\alpha\frac{m_1}{m}}{k+\alpha}$

**Добавление шума**

In [ ]:
def add_noise(series, noise_level):
    return series * (1 + noise_level * np.random.randn(len(series)))

Применим к нашим данным.

In [ ]:
df = sns.load_dataset("titanic")
df['age_noisy'] = add_noise(df['age'], noise_level=0.05)
df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,age_noisy
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,22.366383
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,38.361971
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,27.060417
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,34.519084
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,36.303880


**Кодирование по предыдущим объектам (CatBoost)**

- есть в CatBoost
- одна категория в обучении кодируется по-разному, а на контроле фиксировано

In [ ]:
gb = data.groupby(name)
data[name + '_cb'] = (gb['target'].cumsum() - data['target']) / gb.cumcount()

In [ ]:
df['sex' + '_cb'] = (
    (df.groupby('sex')['survived'].cumsum() - df['survived']) /
    df.groupby('sex').cumcount()
).fillna(df['survived'].mean())
df.head(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,age_noisy,alive_cb,sex_cb
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,22.366383,0.383838,0.383838
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,38.361971,0.383838,0.383838
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,27.060417,1.000000,1.000000
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,34.519084,1.000000,1.000000
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,36.303880,0.000000,0.000000


**Статистики**

Примеры кодирования по другим статистикам для бинарного целевого вектора.

$(y_1,...,y_k), k_1=y_1+...+y_k, k_0=k-k_1$

$mean=\frac{k_1}{k}$

$differ=k_1-k_0$

$differlog=\frac{logk_1}{logk_0}$

$normdiffer=\frac{k_1-k_0}{k}$

<h2><b>Итог</b></h2>

- Качественная трансформация данных является обязательным этапом перед моделированием. Ошибки на этом этапе напрямую снижают точность и интерпретируемость результатов.

- Предобработка данных может быть трудозатратной (до 90% времени).

- Нормировка нужна в тех случаях, когда признаки измеряются в разных шкалах.

- Если переменная категориальная, её нельзя оставлять в первоначальном виде и необходимо корректно кодировать.

- Главный вопрос, который нужно задать перед обработкой: «Почему в данных есть это?»

**Дополнительные материалы**


https://www.kaggle.com/mlisovyi/9-ways-to-treat-categorical-features-updated#

https://www.kaggle.com/ogrellier/python-target-encoding-for-categorical-features#

https://www.kaggle.com/vprokopev/mean-likelihood-encodings-a-comprehensive-study#